# Standard YOLO26 video experiment

Run the pretrained YOLO26 nano model on a soccer video. This is a generic COCO detector; it has not been trained on our soccer annotations. The first model load may download `yolo26n.pt` from Ultralytics, so an internet connection may be required.


## Before running

Select a Python kernel in VS Code or Jupyter. Install the project dependencies into that same environment with `python -m pip install -r requirements-yolo.txt ipykernel`, then restart the kernel. Put your soccer clip in `data/` (it is not included in this checkout). Run the cells in order. The first run defaults to 30 frames; set `MAX_FRAMES = None` for the complete video. Running again replaces this notebook's previous output files.


In [ ]:
from pathlib import Path
import csv
import math
import cv2
from tqdm.auto import tqdm
from ultralytics import YOLO
from IPython.display import display, Image, Video

# Works when the kernel starts in the project root or in notebooks/.
cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (cwd, *cwd.parents) if (p / 'requirements-yolo.txt').is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Open the project folder in VS Code and select a Python kernel before running.')
print('Project folder:', PROJECT_ROOT)


In [ ]:
IMAGE_SIZE_DEFAULT = 640
OUTPUT_SUBDIR = 'yolo26-standard'
# Put the soccer video in data/ or set an absolute path below.
# Leave VIDEO_PATH as None to automatically use the only video in data/.
VIDEO_PATH = None  # Example: PROJECT_ROOT / 'data' / 'match.mp4'
CONFIDENCE = 0.25
IMAGE_SIZE = IMAGE_SIZE_DEFAULT
MAX_FRAMES = 30  # Quick test. Change to None to process the full video.
PREVIEW_EVERY = 30  # Show the first frame and then every 30th frame.

if VIDEO_PATH is None:
    candidates = [p for p in (PROJECT_ROOT / 'data').iterdir() if p.suffix.lower() in {'.mp4', '.mov', '.avi', '.mkv'}]
    if len(candidates) != 1:
        raise FileNotFoundError(f'Put one soccer video in {PROJECT_ROOT / "data"}, or set VIDEO_PATH explicitly. Found {len(candidates)} videos.')
    VIDEO_PATH = candidates[0]
VIDEO_PATH = Path(VIDEO_PATH).expanduser()
if not VIDEO_PATH.is_absolute():
    VIDEO_PATH = PROJECT_ROOT / VIDEO_PATH
if not VIDEO_PATH.is_file():
    raise FileNotFoundError(f'Video not found: {VIDEO_PATH}')
if not 0 < CONFIDENCE <= 1 or IMAGE_SIZE <= 0:
    raise ValueError('Use confidence between 0 and 1 and a positive image size.')
if MAX_FRAMES is not None and (not isinstance(MAX_FRAMES, int) or MAX_FRAMES < 1):
    raise ValueError('MAX_FRAMES must be a positive integer or None.')
if not isinstance(PREVIEW_EVERY, int) or PREVIEW_EVERY < 1:
    raise ValueError('PREVIEW_EVERY must be a positive integer.')
OUTPUT_DIR = PROJECT_ROOT / 'runs' / OUTPUT_SUBDIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_VIDEO = OUTPUT_DIR / f'{VIDEO_PATH.stem}_annotated.mp4'
OUTPUT_CSV = OUTPUT_DIR / f'{VIDEO_PATH.stem}_detections.csv'
print('Input:', VIDEO_PATH)
print('Outputs:', OUTPUT_DIR)


## Load the pretrained model


In [ ]:
model = YOLO('yolo26n.pt')
print('Classes:', model.names)


## Run detection

Each sampled preview shows the model’s predicted boxes. The CSV includes all classes returned by the generic model.


In [ ]:
capture = cv2.VideoCapture(str(VIDEO_PATH))
if not capture.isOpened():
    capture.release()
    raise RuntimeError(f'Could not open video: {VIDEO_PATH}')
writer = None
frame_count = 0
detection_count = 0
try:
    width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = capture.get(cv2.CAP_PROP_FPS)
    if width <= 0 or height <= 0:
        raise RuntimeError('Could not determine video dimensions.')
    if not math.isfinite(fps) or fps <= 0:
        fps = 30.0
        print('Video FPS was unavailable; using 30 FPS for output.')
    total = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
    total = min(total, MAX_FRAMES) if total > 0 and MAX_FRAMES is not None else (total or MAX_FRAMES)
    writer = cv2.VideoWriter(str(OUTPUT_VIDEO), cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))
    if not writer.isOpened():
        raise RuntimeError(f'Could not create output video: {OUTPUT_VIDEO}')
    with OUTPUT_CSV.open('w', newline='', encoding='utf-8') as output:
        rows = csv.writer(output)
        rows.writerow(['frame', 'class_id', 'class_name', 'confidence', 'x1', 'y1', 'x2', 'y2'])
        with tqdm(total=total, desc='Processing video', unit='frame') as progress:
            while MAX_FRAMES is None or frame_count < MAX_FRAMES:
                ok, frame = capture.read()
                if not ok:
                    break
                result = model.predict(source=frame, conf=CONFIDENCE, imgsz=IMAGE_SIZE, verbose=False)[0]
                annotated = result.plot()
                writer.write(annotated)
                if frame_count % PREVIEW_EVERY == 0:
                    ok_image, encoded = cv2.imencode('.jpg', annotated)
                    if ok_image:
                        print(f'Frame {frame_count}')
                        display(Image(data=encoded.tobytes(), width=900))
                if result.boxes is not None:
                    for box in result.boxes:
                        class_id = int(box.cls[0].item())
                        x1, y1, x2, y2 = box.xyxy[0].tolist()
                        rows.writerow([frame_count, class_id, result.names[class_id], float(box.conf[0].item()), x1, y1, x2, y2])
                        detection_count += 1
                frame_count += 1
                progress.update(1)
finally:
    capture.release()
    if writer is not None:
        writer.release()
if frame_count == 0:
    raise RuntimeError('No frames could be read from the video.')
print(f'Processed {frame_count} frames; found {detection_count} detections.')
print('Annotated video:', OUTPUT_VIDEO)
print('Detections CSV:', OUTPUT_CSV)


## View the result

Annotated sample frames appear during processing, and the MP4 and detection CSV are saved in `runs/`. Some notebook viewers cannot play OpenCV's MP4 codec inline; if the player is blank, open the saved MP4 in a video player. Frames in the CSV are zero-indexed and box coordinates are in original video pixels.


In [ ]:
display(Video(str(OUTPUT_VIDEO), embed=False))
print('If the inline player cannot play this file, open it directly:', OUTPUT_VIDEO)
